# Módulo 10 — Diseño de Mecanismos y Subastas

**Objetivos**: Demostrar el teorema de equivalencia de ingresos numéricamente. Implementar el mecanismo de Myerson para la subasta óptima. Introducir el problema del principal-agente.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import uniform
print('Entorno listo.')

## 1. Subastas de primer y segundo precio

**Primer precio**: El ganador (mayor puja) paga su propia puja. Estrategia de equilibrio con n pujadores y valoraciones Uniform[0,1]: b*(v) = v·(n−1)/n

**Vickrey (segundo precio)**: El ganador paga la segunda puja más alta. Estrategia dominante: pujar v (verdad).

**Teorema de Equivalencia de Ingresos**: con valoraciones iid, ambos formatos generan el mismo ingreso esperado para el vendedor.

In [ ]:
def first_price_auction(valuations):
    """Simula una subasta de primer precio. Retorna (ganador, precio)."""
    n = len(valuations)
    bids = [v * (n - 1) / n for v in valuations]  # equilibrio de Bayes-Nash
    winner = np.argmax(bids)
    price = bids[winner]
    return winner, price

def vickrey_auction(valuations):
    """Simula una subasta de Vickrey. Retorna (ganador, precio)."""
    winner = np.argmax(valuations)
    sorted_vals = sorted(valuations, reverse=True)
    price = sorted_vals[1] if len(sorted_vals) > 1 else 0
    return winner, price

# Ejemplo de una ronda
np.random.seed(42)
vals = np.random.uniform(0, 1, 4)
print(f'Valoraciones: {np.round(vals, 3)}')
w1, p1 = first_price_auction(vals)
w2, p2 = vickrey_auction(vals)
print(f'1er precio: ganador=J{w1+1}, precio={p1:.3f}')
print(f'Vickrey:    ganador=J{w2+1}, precio={p2:.3f}')

## 2. Equivalencia de ingresos — demostración numérica

Con n pujadores con valoraciones Uniform[0,1], el ingreso esperado del vendedor es (n-1)/(n+1) para ambos formatos.

In [ ]:
def simulate_revenue(n_bidders, n_rounds=50000, v_max=1.0):
    """Simula n_rounds rondas y retorna (rev_fp_mean, rev_vk_mean, se_fp, se_vk)."""
    fp_revs = []
    vk_revs = []
    for _ in range(n_rounds):
        vals = np.random.uniform(0, v_max, n_bidders)
        _, p1 = first_price_auction(vals)
        _, p2 = vickrey_auction(vals)
        fp_revs.append(p1)
        vk_revs.append(p2)
    return (np.mean(fp_revs), np.mean(vk_revs),
            np.std(fp_revs)/np.sqrt(n_rounds), np.std(vk_revs)/np.sqrt(n_rounds))

print('n | Rev. 1er precio | Rev. Vickrey | Teórico (n-1)/(n+1)')
print('-' * 55)
for n in [2, 3, 5, 10]:
    fp, vk, se_fp, se_vk = simulate_revenue(n, n_rounds=20000)
    teorico = (n-1)/(n+1)
    print(f'{n:2d} | {fp:.4f} ± {se_fp:.4f}  | {vk:.4f} ± {se_vk:.4f} | {teorico:.4f}')

In [ ]:
# Gráfico: ingresos vs. número de pujadores
ns = range(2, 16)
fp_means, vk_means, teoricos = [], [], []

for n in ns:
    fp, vk, _, _ = simulate_revenue(n, n_rounds=10000)
    fp_means.append(fp)
    vk_means.append(vk)
    teoricos.append((n-1)/(n+1))

plt.figure(figsize=(9, 4.5))
plt.plot(ns, fp_means,  '#1a3a5c', marker='o', ms=5, lw=2, label='1er precio (simulado)')
plt.plot(ns, vk_means,  '#b85c00', marker='s', ms=5, lw=2, label='Vickrey (simulado)')
plt.plot(ns, teoricos,  '#2d7a50', ls='--', lw=2, label='Teórico (n-1)/(n+1)')
plt.xlabel('Número de pujadores (n)')
plt.ylabel('Ingreso esperado del vendedor')
plt.title('Equivalencia de ingresos: 1er precio vs. Vickrey')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 3. Mecanismo de Myerson — subasta óptima

Para valoraciones Uniform[0,1], el mecanismo de Myerson establece un precio de reserva óptimo r*. Con n=1 comprador y F=Uniform[0,1], r* = 0.5.

La **función de virtual value**: ψ(v) = v − (1−F(v))/f(v) = 2v − 1 (para Uniform[0,1]).

In [ ]:
def virtual_value_uniform(v):
    """Virtual value para F = Uniform[0,1]."""
    return 2 * v - 1  # ψ(v) = v - (1-v)/1

def myerson_optimal_auction(valuations):
    """
    Aplica el mecanismo de Myerson con F=Uniform[0,1].
    Asigna al pujador con mayor virtual value positivo.
    Retorna (ganador_idx o None, precio).
    """
    vv = [virtual_value_uniform(v) for v in valuations]
    best_idx = np.argmax(vv)
    if vv[best_idx] < 0:
        return None, 0  # No se asigna (precio de reserva implícito)
    # Precio: virtual value = 0 → v = 0.5 (precio de reserva)
    # Más precisamente, el ganador paga el mínimo v que le haría ganar
    price = max(0.5, sorted(vv, reverse=True)[1] if len(vv) > 1 and sorted(vv, reverse=True)[1] > 0 else 0.5)
    price = min(valuations[best_idx], price)  # no puede pagar más que su valoración
    return best_idx, price

# Comparar ingresos
n_bidders = 2
n_rounds = 30000
fp_rev, vk_rev, myo_rev = [], [], []

for _ in range(n_rounds):
    vals = np.random.uniform(0, 1, n_bidders)
    _, p1 = first_price_auction(vals)
    _, p2 = vickrey_auction(vals)
    _, p_myo = myerson_optimal_auction(vals)
    fp_rev.append(p1); vk_rev.append(p2); myo_rev.append(p_myo)

print(f'n={n_bidders} pujadores, F=Uniform[0,1]:')
print(f'  1er precio:  {np.mean(fp_rev):.4f}')
print(f'  Vickrey:     {np.mean(vk_rev):.4f}')
print(f'  Myerson opt: {np.mean(myo_rev):.4f}')
print(f'  Teórico Vickrey/1er precio: {(n_bidders-1)/(n_bidders+1):.4f}')

## 4. Problema del Principal-Agente (intro)

El problema del principal-agente: el principal (contratante) diseña un contrato (w, esfuerzo) para el agente (trabajador) cuyo esfuerzo no es observable.

**Caso simple**: esfuerzo e ∈ {bajo, alto}, pago w, utilidad del agente: U = w − c(e), utilidad de reserva Ū = 0.

In [ ]:
# Modelo simple principal-agente
# Producción: y(e) alta con p_h si esfuerzo alto, p_l si bajo
# Principal diseña (w_h, w_l) para cuando el output es alto o bajo

p_h = 0.8  # prob. output alto si esfuerzo alto
p_l = 0.3  # prob. output alto si esfuerzo bajo
c_e = 0.2  # coste del esfuerzo alto
y_h = 10   # output alto
y_l = 2    # output bajo
U_res = 0  # utilidad de reserva del agente

# Condiciones:
# IR: p_h * w_h + (1-p_h) * w_l - c_e >= U_res  (participación con esfuerzo alto)
# IC: p_h * w_h + (1-p_h) * w_l - c_e >= p_l * w_h + (1-p_l) * w_l  (incentivo a esfuerzo alto)

# Con información perfecta (benchmark): el principal extrae todo el excedente
# w_SB: contrato de segundo-mejor (información asimétrica)

# Resolvemos: IC y IR binding → dos ecuaciones en dos incógnitas
# IC: (p_h - p_l)*(w_h - w_l) = c_e → w_h - w_l = c_e / (p_h - p_l)
# IR: p_h*w_h + (1-p_h)*w_l = U_res + c_e

delta = c_e / (p_h - p_l)
# p_h*(w_l + delta) + (1-p_h)*w_l = U_res + c_e
# w_l + p_h*delta = U_res + c_e
w_l_SB = U_res + c_e - p_h * delta
w_h_SB = w_l_SB + delta

profit_SB = p_h * (y_h - w_h_SB) + (1-p_h) * (y_l - w_l_SB)
print('Contrato de segundo-mejor (información asimétrica):')
print(f'  w_h (output alto) = {w_h_SB:.3f}')
print(f'  w_l (output bajo) = {w_l_SB:.3f}')
print(f'  Beneficio del principal = {profit_SB:.3f}')

# Benchmark: información perfecta (FB)
w_FB = U_res + c_e
profit_FB = p_h * (y_h - w_FB) + (1-p_h) * (y_l - w_FB)
print(f'\nBenchmark first-best (información perfecta):')
print(f'  w_FB = {w_FB:.3f}')
print(f'  Beneficio del principal = {profit_FB:.3f}')
print(f'\nCosto de la información asimétrica: {profit_FB - profit_SB:.3f}')

## Ejercicios

**Ejercicio 1**: Con 5 pujadores y valoraciones Uniform[0, 100], ¿cuál es el precio de reserva óptimo de Myerson? Verifica numéricamente que aumentar el precio de reserva de 0 a 50 incrementa el ingreso esperado.

**Ejercicio 2**: Implementa el mecanismo de Vickrey para 3 bienes idénticos (subasta multi-unidad). Generaliza la función `vickrey_auction` para que el vendedor ponga a subasta k unidades y los k mejores pujadores pagan el (k+1)-ésimo precio.

In [ ]:
# Tu código aquí
